In [ ]:
pip install gurobipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 45.6 MB/s eta 0:00:00


In [ ]:
# Original base travel distances
base_travel_distance = {
    (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60,
    (2, 3): 35, (2, 4): 45, (2, 5): 55,
    (3, 4): 25, (3, 5): 30,
    (4, 5): 20, (0, 5): 35, (0, 4): 40, (0, 3): 45, (0, 2): 50, (0, 1): 55
}

# Adding reverse distances
for (i, j), distance in list(base_travel_distance.items()):
    if (j, i) not in base_travel_distance:
        base_travel_distance[(j, i)] = distance

# Now base_travel_distance contains both (i, j) and (j, i) pairs
print(base_travel_distance)


{(1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60, (2, 3): 35, (2, 4): 45, (2, 5): 55, (3, 4): 25, (3, 5): 30, (4, 5): 20, (0, 5): 35, (0, 4): 40, (0, 3): 45, (0, 2): 50, (0, 1): 55, (2, 1): 30, (3, 1): 40, (4, 1): 50, (5, 1): 60, (3, 2): 35, (4, 2): 45, (5, 2): 55, (4, 3): 25, (5, 3): 30, (5, 4): 20, (5, 0): 35, (4, 0): 40, (3, 0): 45, (2, 0): 50, (1, 0): 55}


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and Indices
n = 6  # Number of Points (node 0 is warehouse)
K = 3  # Number of Products
np.random.seed(6)

# Base travel distances between specific nodes (in km)
base_travel_distance = {
    (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60,
    (2, 3): 35, (2, 4): 45, (2, 5): 55, (3, 4): 25, (3, 5): 30,
    (4, 5): 20, (0, 5): 35, (0, 4): 40, (0, 3): 45, (0, 2): 50, (0, 1): 55,
    (2, 1): 30, (3, 1): 40, (4, 1): 50, (5, 1): 60, (3, 2): 35,
    (4, 2): 45, (5, 2): 55, (4, 3): 25, (5, 3): 30, (5, 4): 20, (5, 0): 35,
    (4, 0): 40, (3, 0): 45, (2, 0): 50, (1, 0): 55
}

# Convert distances to travel times in hours (assuming 50 km/h speed)
base_travel_time = {k: v / 50 for k, v in base_travel_distance.items()}  # Time in hours

# Adding random variability to travel times (in hours)
travel_times = {(i, j): base_travel_time.get((i, j), 0) + np.random.uniform(-0.5, 0.5)
                for i in range(n) for j in range(n)}

# Delay time setup (in seconds, converted to hours)
base_delay_time = 600  # Base delay time in seconds
delays = [base_delay_time / 3600 + np.random.uniform(0.1, 0.2) if np.random.rand() < 0.1 else base_delay_time / 3600
          for _ in range(n)]  # Delay time in hours

# Shelf life setup for three produce types (in hours)
base_shelf_life = {'Apples': 720, 'Bananas': 168, 'Tomatoes': 336}  # in hours
shelf_life = [base_shelf_life['Apples'] * (1 + np.random.uniform(-0.1, 0.1)),
              base_shelf_life['Bananas'] * (1 + np.random.uniform(-0.1, 0.1)),
              base_shelf_life['Tomatoes'] * (1 + np.random.uniform(-0.1, 0.1))]

# Required shelf life setup (in hours)
required_shelf_life = [20 * 24, 5 * 24, 4 * 24]  # in hours

# Priority customers and other parameters
Q = np.random.uniform(0.5, 1, K)
P = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
alpha = 0.5

# Create model
model = gp.Model('Perishable Produce Transportation')

# Decision Variables
x = model.addVars(n, n, lb=0, ub=1, vtype=GRB.BINARY, name="x")  # Route selection (0 if no route, 1 if part of route)
u = model.addVars(n, lb=0, ub=n-1, vtype=GRB.CONTINUOUS, name="u")  # MTZ subtour elimination

# Constraints
# Ensure starting from node 0
model.addConstr(sum(x[0, j] for j in range(1, n)) == 1, name='start_at_0')  # Start from node 0
model.addConstr(sum(x[i, 0] for i in range(1, n)) == 1, name='return_to_0')  # Return to node 0

# Each intermediate node must be entered and exited exactly once
model.addConstrs((sum(x[i, j] for j in range(n) if j != i) == 1 for i in range(1, n)), name='enter_once')
model.addConstrs((sum(x[j, i] for j in range(n) if j != i) == 1 for i in range(1, n)), name='exit_once')


# No self-loops
model.addConstrs((x[i, i] == 0 for i in range(n)), name='const2')

# MTZ Constraints (subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1 for i in range(1, n) for j in range(1, n) if i != j), name='const4')

# In-transit time constraint for each product
for k in range(K):
    model.addConstr(
        sum((delays[i] + travel_times[i, j]) * x[i, j] for i in range(n) for j in range(n))
        <= Q[k] * shelf_life[k] - required_shelf_life[k],
        name=f'const5_{k}'
    )

# Objective function: Minimize cost plus priority term
cost_term = sum(travel_times.get((i, j), 0) * x[i, j] for i in range(n) for j in range(n))  # Travel cost in km
priority_term = sum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

model.setObjective(cost_term, GRB.MINIMIZE)  # Minimize total cost including priority

# Optimize model
model.optimize()

# Display results
if model.status == GRB.OPTIMAL:
    print("Optimal objective value:", model.objVal)
    for i in range(n):
        for j in range(n):
            if x[i, j].X > 0.5:  # Route is used in the solution
                print(f"Route from {i} to {j} is included in the optimal solution.")
else:
    print("No optimal solution found.")


# In-sample cost calculation
if model.status == GRB.OPTIMAL:
    in_sample_cost = sum(travel_times.get((i, j), 0) * x[i, j].X for i in range(n) for j in range(n))
    in_sample_priority_cost = sum((P[i] + P[j]) * alpha * x[i, j].X for i in range(n) for j in range(n))
    total_in_sample_cost = in_sample_cost + in_sample_priority_cost
    print("In-sample travel cost:", in_sample_cost)
    print("In-sample priority cost:", in_sample_priority_cost)
    print("Total in-sample cost:", total_in_sample_cost)
## Increase variability for out-of-sample travel times and delays
out_of_sample_travel_times = {
    (i, j): travel_times[i, j] + np.random.uniform(-2, 2) for i in range(n) for j in range(n)  # Larger variability
}

out_of_sample_delays = [
    delay + np.random.uniform(-0.2, 0.2) for delay in delays  # Larger variability in delays
]


# Calculate out-of-sample cost based on the modified travel times and delays
out_of_sample_cost = sum(out_of_sample_travel_times.get((i, j), 0) * x[i, j].X for i in range(n) for j in range(n))
out_of_sample_priority_cost = sum((P[i] + P[j]) * alpha * x[i, j].X for i in range(n) for j in range(n))
total_out_of_sample_cost = out_of_sample_cost



print("Out-of-sample travel cost:", out_of_sample_cost)
print("Out-of-sample priority cost:", out_of_sample_priority_cost)
print("Total out-of-sample cost:", total_out_of_sample_cost)


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 41 rows, 42 columns and 234 nonzeros
Model fingerprint: 0x529582cd
Variable types: 6 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [7e-02, 6e+00]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 8e+01]
Found heuristic solution: objective 4.0251084
Presolve removed 10 rows and 9 columns
Presolve time: 0.00s
Presolved: 31 rows, 33 columns, 180 nonzeros
Variable types: 5 continuous, 28 integer (28 binary)

Root relaxation: objective 3.179782e+00, 17 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    3.1

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and Indices
n = 6  # Number of Points (node 0 is warehouse)
K = 3  # Number of Products
np.random.seed(6)

# Base travel distances between specific nodes (in km)
base_travel_distance = {
    (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60,
    (2, 3): 35, (2, 4): 45, (2, 5): 55, (3, 4): 25, (3, 5): 30,
    (4, 5): 20, (0, 5): 35, (0, 4): 40, (0, 3): 45, (0, 2): 50, (0, 1): 55,
    (2, 1): 30, (3, 1): 40, (4, 1): 50, (5, 1): 60, (3, 2): 35,
    (4, 2): 45, (5, 2): 55, (4, 3): 25, (5, 3): 30, (5, 4): 20, (5, 0): 35,
    (4, 0): 40, (3, 0): 45, (2, 0): 50, (1, 0): 55
}

# Convert distances to travel times in hours (assuming 50 km/h speed)
base_travel_time = {k: v / 50 for k, v in base_travel_distance.items()}  # Time in hours

# Adding random variability to travel times (in hours)
travel_times = {(i, j): base_travel_time.get((i, j), 0) + np.random.uniform(-0.5, 0.5)
                for i in range(n) for j in range(n)}


# Delay time setup (in seconds, converted to hours)
base_delay_time = 600  # Base delay time in seconds
delays = [base_delay_time / 3600 + np.random.uniform(0.1, 0.2) if np.random.rand() < 0.1 else base_delay_time / 3600
          for _ in range(n)]  # Delay time in hours



# Define robust uncertainty bounds for travel times and delays (in hours)
T_min = {k: max(0, base_travel_time[k] - 0.3) for k in base_travel_time}  # Lower bound of travel time (in hours)
T_max = {k: base_travel_time[k] + 0.3 for k in base_travel_time}  # Upper bound of travel time (in hours)

# Base delay time and robust uncertainty bounds for delays (in hours)
base_delay_time = 600 / 3600  # Base delay time in hours (converted from seconds)
R_min = [base_delay_time - 0.06 if base_delay_time > 0.042 else 0 for _ in range(n)]  # Lower bound of delay (in hours)
R_max = [base_delay_time + 0.06 for _ in range(n)]  # Upper bound of delay (in hours)

# Shelf life setup for three produce types (in hours)
base_shelf_life = {'Apples': 720, 'Bananas': 168, 'Tomatoes': 336}  # in hours
shelf_life = [base_shelf_life['Apples'] * (1 + np.random.uniform(-0.1, 0.1)),
              base_shelf_life['Bananas'] * (1 + np.random.uniform(-0.1, 0.1)),
              base_shelf_life['Tomatoes'] * (1 + np.random.uniform(-0.1, 0.1))]

# Required shelf life setup (in hours)
required_shelf_life = [12 * 24, 5 * 24, 4 * 24]  # in hours

# Priority customers and other parameters
Q = np.random.uniform(0.5, 1, K)
P = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
alpha = 0.5

# Create model
model = gp.Model('Perishable Produce Transportation')

# Decision Variables
x = model.addVars(n, n, lb=0, ub=1, vtype=GRB.BINARY, name="x")  # Route selection (0 if no route, 1 if part of route)
u = model.addVars(n, lb=0, ub=n-1, vtype=GRB.CONTINUOUS, name="u")  # MTZ subtour elimination

# Constraints
# Ensure starting from node 0
model.addConstr(sum(x[0, j] for j in range(1, n)) == 1, name='start_at_0')  # Start from node 0
model.addConstr(sum(x[i, 0] for i in range(1, n)) == 1, name='return_to_0')  # Return to node 0

# Each intermediate node must be entered and exited exactly once
model.addConstrs((sum(x[i, j] for j in range(n) if j != i) == 1 for i in range(1, n)), name='enter_once')
model.addConstrs((sum(x[j, i] for j in range(n) if j != i) == 1 for i in range(1, n)), name='exit_once')


# No self-loops
model.addConstrs((x[i, i] == 0 for i in range(n)), name='const2')

# MTZ Constraints (subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1 for i in range(1, n) for j in range(1, n) if i != j), name='const4')

# In-transit time constraint for each product
# Robust in-transit time constraint with uncertainty for each product (in hours)
for k in range(K):
    model.addConstr(
        gp.quicksum((R_max[i] + T_max.get((i, j), 0)) * x[i, j] for i in range(n) for j in range(n))
        <= Q[k] * shelf_life[k] - required_shelf_life[k],
        name=f'robust_time_constraint_k_{k}'
    )
# Objective function: Minimize cost plus priority term
cost_term = sum(travel_times.get((i, j), 0) * x[i, j] for i in range(n) for j in range(n))  # Travel cost in km
priority_term = sum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

model.setObjective(cost_term, GRB.MINIMIZE)  # Minimize total cost including priority

# Optimize model
model.optimize()

# Display results
if model.status == GRB.OPTIMAL:
    print("Optimal objective value:", model.objVal)
    for i in range(n):
        for j in range(n):
            if x[i, j].X > 0.5:  # Route is used in the solution
                print(f"Route from {i} to {j} is included in the optimal solution.")
else:
    print("No optimal solution found.")


# In-sample cost calculation
if model.status == GRB.OPTIMAL:
    in_sample_cost = sum(travel_times.get((i, j), 0) * x[i, j].X for i in range(n) for j in range(n))
    in_sample_priority_cost = sum((P[i] + P[j]) * alpha * x[i, j].X for i in range(n) for j in range(n))
    total_in_sample_cost = in_sample_cost + in_sample_priority_cost
    print("In-sample travel cost:", in_sample_cost)
    print("In-sample priority cost:", in_sample_priority_cost)
    print("Total in-sample cost:", total_in_sample_cost)

## Increase variability for out-of-sample travel times and delays
out_of_sample_travel_times = {
    (i, j): travel_times[i, j] + np.random.uniform(-2, 2) for i in range(n) for j in range(n)  # Larger variability
}

out_of_sample_delays = [
    delay + np.random.uniform(-0.2, 0.2) for delay in delays  # Larger variability in delays
]


# Calculate out-of-sample cost based on the modified travel times and delays
out_of_sample_cost = sum(out_of_sample_travel_times.get((i, j), 0) * x[i, j].X for i in range(n) for j in range(n))
out_of_sample_priority_cost = sum((P[i] + P[j]) * alpha * x[i, j].X for i in range(n) for j in range(n))
total_out_of_sample_cost = out_of_sample_cost + out_of_sample_priority_cost


print("Out-of-sample travel cost:", out_of_sample_cost)
print("Out-of-sample priority cost:", out_of_sample_priority_cost)
print("Total out-of-sample cost:", total_out_of_sample_cost)



Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 41 rows, 42 columns and 234 nonzeros
Model fingerprint: 0x40640543
Variable types: 6 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [2e-01, 6e+00]
  Objective range  [8e-02, 2e+00]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 2e+02]
Found heuristic solution: objective 6.2250026
Presolve removed 9 rows and 7 columns
Presolve time: 0.00s
Presolved: 32 rows, 35 columns, 180 nonzeros
Variable types: 5 continuous, 30 integer (30 binary)

Root relaxation: objective 3.179782e+00, 17 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    3.17

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and Indices
n = 6  # Number of Points (node 0 is warehouse)
K = 3  # Number of Products
np.random.seed(6)

# Base travel distances between specific nodes (in km)
base_travel_distance = {
    (1, 2): 30, (1, 3): 40, (1, 4): 50, (1, 5): 60,
    (2, 3): 35, (2, 4): 45, (2, 5): 55,
    (3, 4): 25, (3, 5): 30,
    (4, 5): 20
}

# Convert distances to travel times in hours (assuming 50 km/h speed)
base_travel_time = {k: v / 50 for k, v in base_travel_distance.items()}  # travel time in hours

# Define robust uncertainty bounds for travel times and delays (in hours)
T_min = {k: max(0, base_travel_time[k] - 0.1) for k in base_travel_time}  # Lower bound of travel time (in hours)
T_max = {k: base_travel_time[k] + 0.1 for k in base_travel_time}  # Upper bound of travel time (in hours)

# Base delay time and robust uncertainty bounds for delays (in hours)
base_delay_time = 600 / 3600  # Base delay time in hours (converted from seconds)
R_min = [base_delay_time - 0.042 if base_delay_time > 0.042 else 0 for _ in range(n)]  # Lower bound of delay (in hours)
R_max = [base_delay_time + 0.042 for _ in range(n)]  # Upper bound of delay (in hours)

# Shelf life setup for three produce types (in hours)
base_shelf_life = {'Apples': 720, 'Bananas': 168, 'Tomatoes': 336}  # Shelf life in hours
shelf_life = [base_shelf_life['Apples'] * (1 + np.random.uniform(-0.01, 0.01)),
              base_shelf_life['Bananas'] * (1 + np.random.uniform(-0.0, 0.01)),
              base_shelf_life['Tomatoes'] * (1 + np.random.uniform(-0.01, 0.01))]

# Required shelf life setup (should be less than initial shelf life)
required_shelf_life = [10 * 24, 5 * 24, 7 * 24]  # in hours

# Priority customers and other parameters
Q = np.random.uniform(0.5, 1, K)
P = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
alpha = 7

# Create model
model = gp.Model('Perishable Produce Transportation with Robust Uncertainty')

# Variables
x = model.addVars(n, n, vtype=GRB.CONTINUOUS)  # Route selection (1 if arc i-j is part of the route)
u = model.addVars(n, lb=0, ub=float('inf'), vtype=GRB.CONTINUOUS)  # MTZ subtour elimination

# Constraints
model.addConstrs((gp.quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), name='const1')  # Each node entered once
model.addConstrs((x[i, j] == 0 for i in range(n) for j in range(n) if i == j), name='const2')  # No self-loops
model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), name='const3')  # Each node exited once

# MTZ Constraints (subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1 for i in range(n) for j in range(n) if i != j), name='const4')

# Robust in-transit time constraint with uncertainty for each product (in hours)
for k in range(K):
    model.addConstr(
        gp.quicksum((R_max[i] + T_max.get((i, j), 0)) * x[i, j] for i in range(n) for j in range(n))
        <= Q[k] * shelf_life[k] - required_shelf_life[k],
        name=f'robust_time_constraint_k_{k}'
    )

# Objective function: Minimize cost plus priority term
cost_term = sum(base_travel_distance.get((i, j), 0) * x[i, j] for i in range(n) for j in range(n))  # Travel cost in km
priority_term = sum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

model.setObjective(cost_term, GRB.MINIMIZE)
# Optimize model
model.optimize()

# Display results
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model.objVal}")

    for i in range(n):
        for j in range(n):
            if x[i, j].X > 0.5:  # Route is used in the solution
                print(f"Route from {i} to {j} is included in the optimal solution.")
else:
    print(f"Optimization was unsuccessful. Status code: {model.status}")


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 51 rows, 42 columns and 276 nonzeros
Model fingerprint: 0x0e09bf6a
Coefficient statistics:
  Matrix range     [2e-01, 6e+00]
  Objective range  [2e+01, 6e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 6 rows and 6 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
Optimization was unsuccessful. Status code: 3


In [ ]:
for i in range(10):
  print(i)

0
1
2
3
4
5
6
7
8
9
